# Análise completa de um caso

### Carregamento do caso

In [15]:
import pandas as pd
import diplo_grafo.config as cfg
from diplo_grafo.sentencing import get_sentences
from diplo_grafo.sentence import Sentence

# Ajuste para a pasta do projeto
base = cfg.SAMPLE_DATA_DIR

# Carrega os CSVs
cases = pd.read_csv(base / "cases.csv")
metadata = pd.read_csv(base / "metadata.csv")
data_dict = pd.read_csv(base / "data_dictionary.csv")

case = cases.iloc[0]

sentences = get_sentences(case)

print(case['case_text'])
print("\n==================================\n")
print("\n|| ".join([s.text for s in sentences]) + " ||\n")

A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparoscopic distal panc

In [16]:
for s in sentences:
    s.tokenize()
    print(s.tokens)
    

['A', '44', 'year-old', 'woman', 'presented', 'with', 'a', '3', 'day', 'history', 'of', 'right', 'flank', 'and', 'lower', 'quadrant', 'abdominal', 'pain', 'associated', 'with', 'nausea', 'and', 'constipation']
['Her', 'past', 'medical', ',', 'family', 'and', 'medication', 'history', 'were', 'otherwise', 'non-contributory', 'and', 'her', 'physical', 'examination', 'was', 'unremarkable']
['She', 'underwent', 'contrast', 'enhanced', 'computed', 'tomography', ',', 'demonstrating', 'a', '6cm', 'cystic', 'lesion', 'between', 'the', 'stomach', 'and', 'body', 'tail', 'of', 'the', 'pancreas', '(', 'Fig', '1', ')']
['She', 'subsequently', 'underwent', 'EUS-FNA', ',', 'which', 'revealed', 'normal', 'pancreatic', 'echotexture', 'and', 'a', 'cyst', 'measuring', '6cm', 'x', '9cm', 'that', 'was', 'free', 'of', 'internal', 'septations', 'or', 'associated', 'masses', '(', 'Fig', '2', ')', 'but', 'compressed', 'the', 'stomach']
['FNA', 'of', 'the', 'cyst', 'demonstrated', 'no', 'evidence', 'of', 'malign

In [17]:
from diplo_grafo.clinical_terms import reconstruct_clinical_tokens

for sentence in sentences:
    sentence.tokenize()
    sentence.tokens = reconstruct_clinical_tokens(sentence.tokens)

for sentence in sentences:
    print(sentence.tokens)

['A', '44', 'year-old', 'woman', 'presented', 'with', 'a', '3', 'day', 'history', 'of', 'right flank', 'and', 'lower quadrant abdominal pain', 'associated', 'with', 'nausea', 'and', 'constipation']
['Her', 'past', 'medical', ',', 'family', 'and', 'medication', 'history', 'were', 'otherwise', 'non-contributory', 'and', 'her', 'physical', 'examination', 'was', 'unremarkable']
['She', 'underwent', 'contrast enhanced computed tomography', ',', 'demonstrating', 'a', '6cm', 'cystic lesion', 'between', 'the', 'stomach', 'and', 'body', 'tail', 'of', 'the', 'pancreas', '(', 'Fig', '1', ')']
['She', 'subsequently', 'underwent', 'EUS-FNA [endoscopic ultrasound-guided fine needle aspiration]', ',', 'which', 'revealed', 'normal', 'pancreatic echotexture', 'and', 'a', 'cyst', 'measuring', '6cm', 'x', '9cm', 'that', 'was', 'free', 'of', 'internal septations', 'or', 'associated masses', '(', 'Fig', '2', ')', 'but', 'compressed', 'the', 'stomach']
['FNA [fine needle aspiration]', 'of', 'the', 'cyst', '

### Construção do Grafo de Conhecimento com Atributos Ricos (Nodes & Edges)

A partir das sentenças com tokens reconstruídos obtidas na célula anterior, estruturamos o Grafo de Conhecimento clínico seguindo o esquema do diagrama Mermaid em `docs/steps.md`, com entidades e relações em inglês (conforme o visualizador do grafo [Diploghraph-Viewer](https://tochafh.github.io/Diploghraph-Viewer/)).

Integramos as expressões regulares desenvolvidas em `extracao_regex.ipynb` para enriquecimento semântico:
* **Atributos de Nós:** Propriedades intrínsecas da entidade (ex: idade e gênero do `Patient`, dimensões e valores laboratoriais quantificados do `Result`).
* **Atributos de Arestas:** Modulações da relação/ação clínica quando fizer sentido semântico:
  * `TAKES`: quantidade/dosagem da medicação (ex: `dosage=40 mg` ou `dosage=300 mg`).
  * `PRESENTS_WITH`: tempo de evolução do sintoma (ex: `duration=3-day history`).
  * `LOCATED_IN`: lateralidade (`laterality=right`) ou aspecto (`aspect=posterior`).
  * `DIAGNOSED_WITH`: status e método de confirmação (`status=confirmed; method=pathology` vs `status=suspected; method=clinical/imaging`).
  * `UNDERWENT_EXAM`: momento temporal do exame (ex: `timeline=POD 4`).

| Entidade no Mermaid (PT) | Tipo no Grafo (EN) | Prefixo ID | Atributos Típicos |
|---|---|---|---|
| Paciente | `Patient` | `P` | `age=...; gender=...` |
| Sintoma | `Symptom` | `S` | (intrínsecos ao sintoma) |
| Doença | `Disease` | `D` | (intrínsecos à condição) |
| Exame | `Exam` | `E` | (técnica/modalidade) |
| Resultado | `Result` | `R` | `size=...`, `value=...; unit=...`, `status=...` |
| PartesCorpo | `Anatomy` | `A` | (região/sistema) |
| Medicação | `Medication` | `M` | (princípio ativo) |
| Efeito | `Effect` | `Ef` | (efeito adverso) |

In [18]:
from diplo_grafo.graph_builder import KnowledgeGraphBuilder
import diplo_grafo.config as cfg

builder = KnowledgeGraphBuilder()
nodes_df, edges_df = builder.build(case, sentences)

proc_dir = cfg.PROCESSED_DATA_DIR
proc_dir.mkdir(parents=True, exist_ok=True)
nodes_df.to_csv(proc_dir / "nodes.csv", index=False)
edges_df.to_csv(proc_dir / "edges.csv", index=False)

print(f"Nós: {len(nodes_df)} | Arestas: {len(edges_df)}")

Nós: 22 | Arestas: 25


In [19]:
print("=== TABELA DE NÓS (nodes.csv) ===")
display(nodes_df.head(20))

print("\n=== TABELA DE ARESTAS COM ATRIBUTOS (edges.csv) ===")
display(edges_df[edges_df["attributes"] != ""])

print("\n=== AMOSTRA GERAL DE ARESTAS ===")
display(edges_df.head(15))


=== TABELA DE NÓS (nodes.csv) ===


,node_id,type,label,attributes
0,P1,Patient,Patient PMC5137649_01,age=44.0; gender=Female
1,S1,Symptom,lower quadrant abdominal pain,
2,A1,Anatomy,right flank,
3,S2,Symptom,nausea,
4,S3,Symptom,constipation,
5,E1,Exam,contrast enhanced computed tomography,
6,R1,Result,cystic lesion,size=6cm
7,A2,Anatomy,stomach,
8,A3,Anatomy,pancreas,
9,E2,Exam,EUS-FNA [endoscopic ultrasound-guided fine nee...,



=== TABELA DE ARESTAS COM ATRIBUTOS (edges.csv) ===


,edge_id,source_id,target_id,relation,attributes
0,e1,P1,S1,PRESENTS_WITH,duration=3-day history
1,e2,S1,A1,LOCATED_IN,laterality=right
2,e3,P1,S2,PRESENTS_WITH,duration=3-day history
3,e4,P1,S3,PRESENTS_WITH,duration=3-day history
18,e19,P1,D1,DIAGNOSED_WITH,status=suspected; method=clinical/imaging
20,e21,P1,D2,DIAGNOSED_WITH,status=confirmed
21,e22,D2,A2,LOCATED_IN,aspect=posterior
22,e23,D2,A4,LOCATED_IN,aspect=posterior
24,e25,P1,D3,DIAGNOSED_WITH,status=confirmed; method=pathology



=== AMOSTRA GERAL DE ARESTAS ===


,edge_id,source_id,target_id,relation,attributes
0,e1,P1,S1,PRESENTS_WITH,duration=3-day history
1,e2,S1,A1,LOCATED_IN,laterality=right
2,e3,P1,S2,PRESENTS_WITH,duration=3-day history
3,e4,P1,S3,PRESENTS_WITH,duration=3-day history
4,e5,P1,E1,UNDERWENT_EXAM,
5,e6,E1,R1,REVEALS,
6,e7,R1,A2,LOCATED_IN,
7,e8,R1,A3,LOCATED_IN,
8,e9,P1,E2,UNDERWENT_EXAM,
9,e10,E2,R2,REVEALS,


In [20]:
from IPython.display import display, Markdown

def generate_mermaid_code(nodes, edges):
    lines = ["```mermaid", "graph LR"]
    for _, r in nodes.iterrows():
        nid = r["node_id"]
        lbl = str(r["label"]).replace('"', "'")
        ntype = r["type"]
        attrs = str(r["attributes"]).replace('"', "'") if pd.notna(r.get("attributes")) and r.get("attributes") else ""
        attr_str = f"<br/><small>{attrs}</small>" if attrs else ""
        
        if ntype == "Patient":
            lines.append(f'    {nid}(("{lbl}{attr_str}"))')
        else:
            lines.append(f'    {nid}["{lbl} <br/> <i>({ntype})</i>{attr_str}"]')

    for _, r in edges.iterrows():
        src = r["source_id"]
        tgt = r["target_id"]
        rel = r["relation"]
        attrs = str(r["attributes"]).replace('"', "'") if pd.notna(r.get("attributes")) and r.get("attributes") else ""
        rel_label = f"{rel}<br/><i>[{attrs}]</i>" if attrs else rel
        lines.append(f'    {src} -- "{rel_label}" --> {tgt}')
    lines.append("```")
    return "\n".join(lines)

mermaid_md = generate_mermaid_code(nodes_df, edges_df)
display(Markdown(mermaid_md))
print(mermaid_md)


```mermaid
graph LR
    P1(("Patient PMC5137649_01<br/><small>age=44.0; gender=Female</small>"))
    S1["lower quadrant abdominal pain <br/> <i>(Symptom)</i>"]
    A1["right flank <br/> <i>(Anatomy)</i>"]
    S2["nausea <br/> <i>(Symptom)</i>"]
    S3["constipation <br/> <i>(Symptom)</i>"]
    E1["contrast enhanced computed tomography <br/> <i>(Exam)</i>"]
    R1["cystic lesion <br/> <i>(Result)</i><br/><small>size=6cm</small>"]
    A2["stomach <br/> <i>(Anatomy)</i>"]
    A3["pancreas <br/> <i>(Anatomy)</i>"]
    E2["EUS-FNA [endoscopic ultrasound-guided fine needle aspiration] <br/> <i>(Exam)</i>"]
    R2["pancreatic echotexture <br/> <i>(Result)</i><br/><small>status=normal</small>"]
    R3["cyst <br/> <i>(Result)</i><br/><small>size=6cm x 9cm</small>"]
    E3["FNA [fine needle aspiration] <br/> <i>(Exam)</i>"]
    R4["extracellular mucin <br/> <i>(Result)</i>"]
    R5["carcinoembryonic antigen <br/> <i>(Result)</i><br/><small>value=12,476.5; unit=ng/ml</small>"]
    R6["carbohydrate antigen <br/> <i>(Result)</i><br/><small>value=6; unit=iu/ml</small>"]
    D1["mucinous pancreatic cystic neoplasm <br/> <i>(Disease)</i>"]
    E4["Intraoperative endoscopy <br/> <i>(Exam)</i>"]
    D2["gastric duplication cyst <br/> <i>(Disease)</i>"]
    A4["posterior wall <br/> <i>(Anatomy)</i>"]
    E5["oesophagogastroduodenoscopy <br/> <i>(Exam)</i>"]
    D3["GDC [gastric duplication cyst] <br/> <i>(Disease)</i>"]
    P1 -- "PRESENTS_WITH<br/><i>[duration=3-day history]</i>" --> S1
    S1 -- "LOCATED_IN<br/><i>[laterality=right]</i>" --> A1
    P1 -- "PRESENTS_WITH<br/><i>[duration=3-day history]</i>" --> S2
    P1 -- "PRESENTS_WITH<br/><i>[duration=3-day history]</i>" --> S3
    P1 -- "UNDERWENT_EXAM" --> E1
    E1 -- "REVEALS" --> R1
    R1 -- "LOCATED_IN" --> A2
    R1 -- "LOCATED_IN" --> A3
    P1 -- "UNDERWENT_EXAM" --> E2
    E2 -- "REVEALS" --> R2
    R2 -- "LOCATED_IN" --> A2
    E2 -- "REVEALS" --> R3
    R3 -- "LOCATED_IN" --> A2
    P1 -- "UNDERWENT_EXAM" --> E3
    E3 -- "REVEALS" --> R3
    E3 -- "REVEALS" --> R4
    E3 -- "REVEALS" --> R5
    E3 -- "REVEALS" --> R6
    P1 -- "DIAGNOSED_WITH<br/><i>[status=suspected; method=clinical/imaging]</i>" --> D1
    P1 -- "UNDERWENT_EXAM" --> E4
    P1 -- "DIAGNOSED_WITH<br/><i>[status=confirmed]</i>" --> D2
    D2 -- "LOCATED_IN<br/><i>[aspect=posterior]</i>" --> A2
    D2 -- "LOCATED_IN<br/><i>[aspect=posterior]</i>" --> A4
    P1 -- "UNDERWENT_EXAM" --> E5
    P1 -- "DIAGNOSED_WITH<br/><i>[status=confirmed; method=pathology]</i>" --> D3
```

```mermaid
graph LR
    P1(("Patient PMC5137649_01<br/><small>age=44.0; gender=Female</small>"))
    S1["lower quadrant abdominal pain <br/> <i>(Symptom)</i>"]
    A1["right flank <br/> <i>(Anatomy)</i>"]
    S2["nausea <br/> <i>(Symptom)</i>"]
    S3["constipation <br/> <i>(Symptom)</i>"]
    E1["contrast enhanced computed tomography <br/> <i>(Exam)</i>"]
    R1["cystic lesion <br/> <i>(Result)</i><br/><small>size=6cm</small>"]
    A2["stomach <br/> <i>(Anatomy)</i>"]
    A3["pancreas <br/> <i>(Anatomy)</i>"]
    E2["EUS-FNA [endoscopic ultrasound-guided fine needle aspiration] <br/> <i>(Exam)</i>"]
    R2["pancreatic echotexture <br/> <i>(Result)</i><br/><small>status=normal</small>"]
    R3["cyst <br/> <i>(Result)</i><br/><small>size=6cm x 9cm</small>"]
    E3["FNA [fine needle aspiration] <br/> <i>(Exam)</i>"]
    R4["extracellular mucin <br/> <i>(Result)</i>"]
    R5["carcinoembryonic antigen <br/> <i>(Result)</i><br/><small>value=12,476.5; unit=ng/ml</small>"]
    R6["carbohydrat